# UAV Neo - Hardware Bringup / Sim-to-Real Test

Code you write in the simulator does not automatically work on the real drone.
This notebook checks the pieces that commonly differ - coordinate frames,
heading, the altitude source, position, command signs and speed - one at a time,
cheapest and safest first. Run it once before trusting a real flight.

Work top to bottom. Each **Tier** says what it proves and what a pass looks like.
If a tier fails, stop and fix it before moving on: a sign or frame mistake caught
here is a one-line change, but in the air it is a crash.

| Tier | Proves | Motors? |
|---|---|---|
| 1 | Frame / heading math (pure Python, no drone) | no |
| 2 | Live telemetry reads correctly as you move the airframe by hand | no |
| 3a | Outgoing velocity and position setpoints have the right sign and scale | no |
| 3b | Motors spin the right way for each command | **yes** |

## Start the drone stack first

Autonomous code publishes setpoints straight to the flight controller, so bring up
the stack **without** the manual-teleop mux (with the mux running it would publish
its own velocity setpoints and fight yours):

```bash
ros2 launch uav_neo_ros2_driver teleop.launch.py manual:=false
```

## SAFETY

- **Props off for every cell except Tier 3b.** Tiers 1-3a never spin motors.
- **Tier 3b spins motors.** Only run it with props off, the frame physically
  restrained, and the safety pilot armed and in OFFBOARD on the RC transmitter.
- The safety layer for autonomous flight is the **RC pilot**: they arm, switch to
  OFFBOARD, and override at any time by switching out of OFFBOARD. There is no mux
  and no bumper dead-man in this path.
- Camera / EdgeTPU bringup is covered separately by `test_async_core_real.ipynb`.


## Tier 1 - frame / heading math (no drone, no ROS)

The real backend converts the flight controller's raw data into the frame the
labs expect (x=right, y=up, z=forward; heading clockwise from north). This tier
imports those conversion helpers and checks them against known orientations. It
needs no drone and no ROS - it runs anywhere the library imports.

**Pass:** the last cell prints `Tier 1 PASS`. Any failed `assert` names the exact
conversion that is wrong.

In [ ]:
import sys, math
import numpy as np
import drone_core

# Put the real backend on the path so we can import its conversion helpers.
_LIB = drone_core.__file__.replace("drone_core.py", "")
sys.path.insert(1, _LIB + "real")
print("library:", _LIB)


In [ ]:
from physics_real import (
    _enu_vector_to_library,
    _enu_rates_to_library,
    _average_orientation_to_attitude,
)


def approx(a, b, tol=1e-3):
    return np.allclose(np.asarray(a, float), np.asarray(b, float), atol=tol)


def yaw_quat(deg):
    # Pure yaw about the ENU up axis -> [w, x, y, z].
    t = math.radians(deg) / 2.0
    return np.array([math.cos(t), 0.0, 0.0, math.sin(t)])


# Body ENU (forward, left, up) -> library (right, up, forward).
assert approx(_enu_vector_to_library(1, 0, 0), [0, 0, 1]), "forward -> +forward"
assert approx(_enu_vector_to_library(0, -1, 0), [1, 0, 0]), "rightward -> +right"
assert approx(_enu_vector_to_library(0, 0, 1), [0, 1, 0]), "up -> +up"

# Body ENU rates -> library (pitch, yaw, roll) rates.
assert approx(_enu_rates_to_library(1, 0, 0), [0, 0, 1]), "roll rate"
assert approx(_enu_rates_to_library(0, 1, 0), [-1, 0, 0]), "pitch rate"
assert approx(_enu_rates_to_library(0, 0, 1), [0, -1, 0]), "yaw rate"

# Attitude: heading is clockwise from north over [0, 360). N=0, E=90, S=180, W=270.
for yaw_enu, expect in [(0, 90), (90, 0), (180, 270), (270, 180)]:
    att = _average_orientation_to_attitude([yaw_quat(yaw_enu)])
    assert approx(att[2], expect), f"heading enu={yaw_enu} -> {att[2]} (want {expect})"
    assert approx(att[0], 0) and approx(att[1], 0), "level pitch/roll"

# q and -q are the same rotation; a raw component mean would cancel. This must not.
q = yaw_quat(90)
att = _average_orientation_to_attitude([q, -q, q, -q])
assert approx(att[2], 0), f"sign-flip heading corrupted: {att[2]}"

# North-boundary guard: yaw must stay strictly below 360 even for hair-negative operands.
for eps in [1e-9, 1e-12, 1e-14, 0.0]:
    att = _average_orientation_to_attitude([yaw_quat(90 + eps)])
    assert 0.0 <= att[2] < 360.0, f"guard failed at eps={eps}: {att[2]}"

print("Tier 1 PASS: vector, rate, attitude, sign-flip, and north-guard all correct")


## Tier 2 - live telemetry (props off, hand-carried)

Build the real drone and read the sensor streams while you move the airframe by
hand. This checks that the frame conversions and the EKF altitude source read
correctly end to end. No arming, no motors - you carry the drone yourself.

In [ ]:
import time

drone = drone_core.create_drone(isSimulation=False)
drone.go_async()   # spins the ROS executor in a background thread (no Xbox START gate)
print("async executor running; waiting for topics to connect...")
time.sleep(3)


### Topic liveness

Confirms the streams the library depends on are up: the IMU, the `/position` and
`/velocity` relays that feed `get_position()`/`get_altitude()`, and the outgoing
`/mavros/setpoint_velocity/cmd_vel` the flight commands publish to. `MISSING` here
usually means the stack was not launched, or was launched without `manual:=false`.

In [ ]:
import subprocess

wanted = ["/mavros/imu/data", "/position", "/velocity",
          "/mavros/setpoint_velocity/cmd_vel"]
out = subprocess.run(["ros2", "topic", "list"], capture_output=True, text=True).stdout
for t in wanted:
    print(("  OK       " if t in out else "  MISSING  ") + t)

### At-rest sanity

Level and still on the bench. Expect: acceleration ~ (0, +9.8, 0) in (right, up,
forward) - gravity on the up axis; angular velocity ~ 0; pitch/roll ~ 0; heading
steady (not jumping - that would mean the attitude averaging is wrong).


In [ ]:
for _ in range(6):
    a = drone.physics.get_linear_acceleration()
    w = drone.physics.get_angular_velocity()
    att = drone.physics.get_attitude()
    print(f"accel(R,U,F)=({a[0]:+5.2f},{a[1]:+5.2f},{a[2]:+5.2f})  "
          f"gyro=({w[0]:+5.2f},{w[1]:+5.2f},{w[2]:+5.2f})  "
          f"pitch={att[0]:+6.1f} roll={att[1]:+6.1f} yaw={att[2]:6.1f}")
    time.sleep(0.5)


### Hand-carry checklist

Interrupt the kernel (stop button) to end the loop. Move the airframe slowly and
check each response:

| Motion | Expected reading |
|---|---|
| Carry **forward** | `vel(R,U,F)` forward (3rd) goes positive |
| Carry **right** | `vel` right (1st) goes positive |
| **Lift up** by hand | `alt` increases (this is the indoor-altitude fix) |
| Rotate **clockwise** (top view) | `heading` increases |
| Point at magnetic **north** | `heading` ~ 0 (the case the old Euler average corrupted) |
| Tilt **nose up** | `pitch` positive |
| Dip **right wing** | `roll` positive |
| Walk **forward** a few meters | `pos` z_north (3rd) increases, x_east (1st) stays flat |

The last column of the loop confirms `get_altitude()` tracks `get_position()` up
(`y_up`) - both come from the same rangefinder-aided EKF pose.


In [ ]:
try:
    while True:
        p = drone.physics.get_position()
        v = drone.physics.get_linear_velocity()
        att = drone.physics.get_attitude()
        alt = drone.physics.get_altitude()
        consistent = abs(alt - p[1]) < 0.05
        print(f"pos(E,U,N)=({p[0]:+5.2f},{p[1]:+5.2f},{p[2]:+5.2f})  "
              f"vel(R,U,F)=({v[0]:+5.2f},{v[1]:+5.2f},{v[2]:+5.2f})  "
              f"heading={att[2]:6.1f}  alt={alt:+5.2f}  alt==pos_up:{consistent}",
              end="\r")
        time.sleep(0.1)
except KeyboardInterrupt:
    print("\nstopped")


## Tier 3a - setpoint sign and scale (props off, disarmed)

The library now publishes flight commands straight to
`/mavros/setpoint_velocity/cmd_vel` (there is no mux in the autonomy path), and it
applies the speed cap itself. This tier commands one axis at a time and reads that
topic back, checking the **sign** and **scale** of the outgoing setpoint. No motors -
this is independent of whether the pilot has armed.

**Pass:** each axis moves the expected component in the expected direction, and the
final cells print `Tier 3a signs PASS` and `speed-scale PASS`.

In [ ]:
import rclpy
from geometry_msgs.msg import TwistStamped

_latest = {"msg": None}
_mon = rclpy.create_node("cmd_vel_monitor")
_mon.create_subscription(
    TwistStamped, "/mavros/setpoint_velocity/cmd_vel",
    lambda m: _latest.__setitem__("msg", m), 10)
rclpy.get_global_executor().add_node(_mon)
time.sleep(0.5)


def read_cmd_vel(hold=1.0):
    # flight_real republishes the last setpoint at 20 Hz, so a short hold suffices.
    # Values are in m/s / rad/s (the library already applied the speed cap).
    time.sleep(hold)
    t = _latest["msg"].twist
    return (t.linear.x, t.linear.y, t.linear.z, t.angular.z)


print("setpoint_velocity monitor attached")

In [ ]:
CMD = 0.4


def axis_test(label, pitch=0, roll=0, yaw=0, throttle=0, expect=""):
    drone.flight.send_pcmd(pitch, roll, yaw, throttle)
    lx, ly, lz, az = read_cmd_vel()
    print(f"{label:22s} -> linear=({lx:+.2f},{ly:+.2f},{lz:+.2f}) m/s  "
          f"angular.z={az:+.2f} rad/s   expect: {expect}")


axis_test("pitch forward (+)", pitch=+CMD, expect="linear.x > 0")
axis_test("roll right (+)", roll=+CMD, expect="linear.y < 0  (setpoint is ENU left-positive)")
axis_test("yaw clockwise (+)", yaw=+CMD, expect="angular.z < 0  (setpoint is ENU CCW-positive)")
axis_test("throttle up (+)", throttle=+CMD, expect="linear.z > 0")
drone.flight.stop()

In [ ]:
# Programmatic sign check (same axes, asserted).
drone.flight.send_pcmd(+CMD, 0, 0, 0)
assert read_cmd_vel()[0] > 0, "pitch: linear.x should be positive"
drone.flight.send_pcmd(0, +CMD, 0, 0)
assert read_cmd_vel()[1] < 0, "roll right: linear.y should be negative"
drone.flight.send_pcmd(0, 0, +CMD, 0)
assert read_cmd_vel()[3] < 0, "yaw CW: angular.z should be negative"
drone.flight.send_pcmd(0, 0, 0, +CMD)
assert read_cmd_vel()[2] > 0, "throttle up: linear.z should be positive"
drone.flight.stop()
print("Tier 3a signs PASS")


### Speed scale

`neo_lab.send_velocity` normalizes a body-frame velocity (m/s) by dividing by
`REAL_MAX_SPEED`, and `flight_real` then scales that normalized command back to m/s
by multiplying by `MAX_SPEED`. Those two constants must match, so a 0.3 m/s request
should appear on `/mavros/setpoint_velocity/cmd_vel` as 0.3 m/s. If they disagree,
every velocity command is silently off by that ratio.

In [ ]:
try:
    import neo_lab
    import flight_real

    class _Shim:
        pass

    shim = _Shim()
    shim.flight = drone.flight

    requested = 0.3
    neo_lab.send_velocity(shim, 0.0, 0.0, requested)   # (v_right, v_up, v_forward) m/s
    lx = read_cmd_vel()[0]
    drone.flight.stop()
    print(f"send_velocity(forward={requested}) -> linear.x={lx:.3f} m/s, "
          f"expect {requested:.3f}\n"
          f"  REAL_MAX_SPEED={neo_lab.REAL_MAX_SPEED}, "
          f"flight_real.MAX_SPEED={flight_real.MAX_SPEED}")
    assert neo_lab.REAL_MAX_SPEED == flight_real.MAX_SPEED, \
        "REAL_MAX_SPEED must equal flight_real.MAX_SPEED"
    assert abs(lx - requested) < 1e-3, "speed scale mismatch"
    print("speed-scale PASS")
except ImportError as e:
    print(f"import failed: {e}")
    print("Check by hand: neo_lab.REAL_MAX_SPEED must equal flight_real.MAX_SPEED (0.5).")

### Position setpoint (goto_position)

`goto_position` publishes an absolute waypoint to `/mavros/setpoint_position/local`
for PX4 to fly to. Props off and disarmed, confirm the setpoint appears with the
values you asked for. Only one setpoint type streams at a time: calling
`goto_position` switches the library to position mode (the velocity stream stops),
and `stop()` switches it back.

**Pass:** the printed x/y/z match the request and the cell prints
`position-setpoint PASS`.

In [ ]:
from geometry_msgs.msg import PoseStamped

_pos_latest = {"msg": None}
_pos_mon = rclpy.create_node("setpoint_pos_monitor")
_pos_mon.create_subscription(
    PoseStamped, "/mavros/setpoint_position/local",
    lambda m: _pos_latest.__setitem__("msg", m), 10)
rclpy.get_global_executor().add_node(_pos_mon)
time.sleep(0.5)

# Ask for a point 1 m east, 1.5 m up, 1 m north (absolute in the EKF frame).
drone.flight.goto_position(1.0, 1.5, 1.0)   # (east, up, north)
time.sleep(1.0)
p = _pos_latest["msg"].pose.position
print(f"setpoint_position/local -> x(east)={p.x:+.2f} y(north)={p.y:+.2f} "
      f"z(up)={p.z:+.2f}")
assert abs(p.x - 1.0) < 1e-3 and abs(p.y - 1.0) < 1e-3 and abs(p.z - 1.5) < 1e-3, \
    "goto_position did not publish the requested point"
drone.flight.stop()   # back to velocity mode
print("position-setpoint PASS")

## Tier 3b - motor direction (props OFF, armed)

**Props off, frame restrained.** The safety pilot arms and holds OFFBOARD on the RC
transmitter; they can override at any time by leaving OFFBOARD. This is the only
check for the `flight_real` roll/yaw sign fixes, which were derived from the gamepad
node rather than measured.

Each command holds for 3 seconds. Confirm the frame *tries* to:

- pitch forward: **nose-down / forward** thrust
- roll right: **right side drops** (left motors spin up)
- yaw clockwise: **nose rotates clockwise** viewed from above
- throttle up: **all motors** spin up together

In [ ]:
sweep = [
    ("pitch forward", dict(pitch=+CMD)),
    ("roll right", dict(roll=+CMD)),
    ("yaw clockwise", dict(yaw=+CMD)),
    ("throttle up", dict(throttle=+CMD)),
]
for label, kw in sweep:
    print(f">>> commanding {label} for 3 s - watch the motors")
    drone.flight.send_pcmd(kw.get("pitch", 0), kw.get("roll", 0),
                           kw.get("yaw", 0), kw.get("throttle", 0))
    time.sleep(3)
    drone.flight.stop()
    time.sleep(1)
print("Tier 3b done")


## Teardown

In [ ]:
drone.flight.stop()
print("stopped. Restart the kernel to fully release the ROS nodes.")
